# Livrable 2 : Datasets, Implémentation des Algorithmes et Plan d'Expérience

## Projet CesiCDP — Résolution du TSPTW par Heuristiques

---

## I. Datasets — Génération des Instances

### 1.1 Source des Données

Dans notre projet, les instances sont entièrement **générées de manière aléatoire**.

Chaque ville est caractérisée par :

- Des coordonnées $(x, y)$ dans un plan cartésien
- Une fenêtre temporelle $[\text{earliest}, \text{latest}]$

Le **dépôt** (ville de départ) est fixé à l'identifiant 0 et possède une fenêtre temporelle très large $[0, 10000]$, ce qui le rend toujours accessible. Toutes les tournées commencent et se terminent à ce point.

Nous réalisons également l'hypothèse suivante :

$$\text{vitesse} = 1 \text{ unité de distance par unité de temps}$$

Donc :

$$\text{temps} = \frac{\text{distance}}{\text{vitesse}} = \frac{\text{distance}}{1} = \text{distance}$$

Cette modélisation suppose une **vitesse constante égale à 1**. Ainsi, toutes les grandeurs sont comparables entre elles, puisque la distance et le temps utilisent la même échelle.

De plus, les coordonnées des villes sont générées aléatoirement dans un carré $[0, 100] \times [0, 100]$, ce qui permet de simuler une répartition géographique.

Grâce à ces coordonnées, nous calculons les distances entre les villes en utilisant la **formule de distance euclidienne** :

$$d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$

### 1.2 Génération Aléatoire des Instances

Les instances sont générées aléatoirement à chaque exécution.

Pour chaque run :

- Un ensemble de villes est généré avec :
  - Des coordonnées aléatoires
  - Des fenêtres temporelles aléatoires $[\text{earliest}, \text{latest}]$, avec $\text{earliest} \in [0, 50]$ et $\text{latest} > \text{earliest}$
- Le nombre de villes $n$ est fixé selon l'expérience (entre 4 et 100)
- Le dépôt est toujours la ville 0
- Les autres villes sont cataloguées de 1 à $n-1$

Lors de l'évaluation d'une tournée :

- Si le véhicule **arrive avant** $\text{earliest}$, il **attend** (autorisé)
- Si l'arrivée **dépasse** $\text{latest}$, une **pénalité de 1000** est appliquée

Certaines arcs entre villes sont **bloquées aléatoirement** (environ 2%). Cela signifie que :

- Certaines connexions ne sont pas autorisées
- Ces contraintes varient à chaque instance

Si une arc bloquée est empruntée, une **pénalité de 1000** est ajoutée au coût total.

Ainsi, le score total est :

$$\text{score} = \text{distance totale} + \text{pénalités}$$

## II. Implémentation des Algorithmes

Notre programme est structuré en plusieurs modules principaux :

| Module | Rôle |
| --- | --- |
| **Ville, Graphe** | Structures de données représentant les villes et le réseau routier |
| **generer_instance()** | Génération d'instances aléatoires (coordonnées, fenêtres temporelles, arcs bloqués) |
| **evaluer_tournee()** | Calcul du score d'une tournée en intégrant les contraintes (fenêtres temporelles et arcs bloqués) |
| **glouton()** | Algorithme heuristique du plus proche voisin (complexité $O(n^2)$) |
| **recuit()** | Algorithme de recuit simulé basé sur des mouvements 2-opt pour améliorer la solution |
| **verifier_solution()** | Vérification de la validité du cycle hamiltonien et du respect des contraintes |
| **Plan d'expérience** | Étude statistique des performances sur différentes tailles d'instances générées aléatoirement |
| **Visualisation & analyse** | Génération de graphiques et comparaison des résultats entre les algorithmes |

### 2.1 Algorithme Glouton (Nearest Neighbor)

L'algorithme glouton repose sur l'heuristique du **plus proche voisin**. À chaque étape, il choisit la ville non visitée la plus proche dont l'arc n'est pas bloquée.

Cet algorithme est rapide, avec une complexité en $O(n^2)$, mais il **ne garantit pas une solution optimale**.

**Pseudo-code :**

```
tour ← [dépôt]

Tant que des villes restent non visitées :
    candidats ← villes accessibles (arcs non bloquées)
    prochain  ← ville la plus proche parmi les candidats
    tour.append(prochain)

tour.append(dépôt)   ← fermeture du cycle hamiltonien
```

Cet algorithme construit donc une solution rapidement, mais ne prend **pas en compte les fenêtres temporelles** lors de la construction du trajet.

### 2.2 Solveur Exact (Branch and Bound)

Le solveur exact implémente une méthode de type **Branch and Bound** à l’aide de la bibliothèque PuLP et du solveur CBC. Cette approche permet de résoudre le problème de manière exacte en recherchant la meilleure solution possible tout en éliminant certaines solutions inutiles afin de réduire le temps de calcul.

Le principe est le suivant :

1. On construit progressivement un chemin (solution partielle)
2. À chaque étape, on calcule un coût partiel
3. Si ce coût est déjà plus mauvais que la meilleure solution trouvée, on arrête d'explorer cette branche (**élagage**)

Ce solveur prend en compte les contraintes du problème :

- Si une arc bloquée est utilisée → pénalité de **+1000**
- Si une fenêtre temporelle n'est pas respectée → pénalité de **+1000**

Ainsi, le solveur exact minimise :

$$\text{score} = \text{distance totale} + \text{pénalités}$$

Cette méthode garantit de trouver la meilleure solution lorsque le temps de calcul est suffisant. Cependant, sa complexité reste très élevée, proche de O(n!) dans le pire cas.

Le temps d’exécution augmente donc très rapidement avec le nombre de villes.

> **Note :** En raison de la complexité élevée, le solveur est utilisé **uniquement pour des petites instances** (généralement $n \leq 10 $).

### 2.3 Recuit Simulé

Le recuit simulé est une **méta-heuristique d'optimisation** qui améliore une solution initiale (ici, celle du glouton). Sa complexité est de $O(n \times k \times \text{itérations})$ avec $k$ = nombre de paliers (lié à $\alpha$).

Il utilise des transformations de type **2-opt**, qui consistent à améliorer un chemin en inversant une partie du trajet entre deux points pour réduire la distance totale.

L'acceptation d'une solution se fait selon le **critère de Metropolis** :

$$P(\text{accepter}) = \exp\left(\frac{-\Delta}{T}\right)$$

où :
- $\Delta$ représente le coût du changement (plus $\Delta$ est grand, plus la nouvelle solution est mauvaise)
- $T$ est la **température** (degré de liberté)
  - Au début, $T$ est élevé : on accepte souvent des solutions moins bonnes pour **explorer** toutes les possibilités
  - À la fin, $T$ est bas : on devient très strict et on n'accepte presque que les meilleures solutions

**Paramètres utilisés :**

| Paramètre | Valeur | Rôle |
| --- | --- | --- |
| $T_{\text{initial}}$ | 200.0 | Exploration large au début |
| $T_{\text{final}}$ | 0.1 | Arrêt de l'algorithme |
| $\alpha$ (alpha) | 0.995 | Refroidissement progressif |
| iterations | 50 | Nombre d'essais par température |

### 2.4 Gestion des Contraintes

Les contraintes sont gérées dans la fonction `evaluer_tournee()` à l'aide de **pénalités**.

**Fenêtres temporelles (Time Windows) :**

- Si le véhicule arrive **avant** $\text{earliest}$ → il **attend** (autorisé)
- Si le véhicule arrive **après** $\text{latest}$ → **pénalité de +1000**

**Arcs bloquées :**

- Si une arc $(i, j)$ est interdite → **pénalité de +1000**

Ainsi, le **score total** est :

$$\text{score} = \text{distance totale} + \text{pénalités}$$

## III. Proposition plan d'Exécution

Dans ce futur plan d'expérience, nous comparerons les performances de l'algorithme glouton et du recuit simulé sur plusieurs tailles d'instances.

**Protocole expérimental :**

- Les instances seront générées de manière aléatoire à chaque exécution
- Pour chaque taille, nous réaliserons **30 runs** afin d'obtenir des résultats représentatifs (moyenne, écart-type, minimum, maximum)
- Une graine aléatoire différente sera utilisée à chaque run : $\text{graine} = \text{seed} \times 13 + n$. Cela garantit la **reproductibilité** tout en assurant des instances variées

## IV. Exemples d'Exécution

### 4.1 Instance Facile — Glouton

Paramètres :

| Paramètre | Valeur |
| --- | --- |
| Température initiale | 200.0 |
| Température finale | 1.0 |
| Facteur de refroidissement | 0.995 |
| Nombre d'itérations par palier | 50 |

Voici les résultats obtenus pour **4 villes** et **0 arcs bloqués** :

**Algorithme Glouton :**
- Cycle : `[0, 2, 3, 1, 0]`
- Score : **1075.59**

**Solveur Exact (Branch and Bound) :**
- Cycle optimal : `[0, 1, 2, 3, 0]`
- Score optimal : **72.62**

**Algorithme Recuit Simulé :**
- Cycle : `[0, 1, 2, 3, 0]`
- Score : **72.62**

Cette instance est considérée comme étant facile pour plusieurs raisons :

Le nombre de villes est faible.
Les fenêtres temporelles ne sont pas contraignantes.
Peu d’arcs sont bloquées.

Le glouton construit une solution rapidement en choisissant toujours la ville la plus proche accessible.
Il obtient un score de 1075.59, ce qui est une mauvaise solution due à une pénalité de 1000.

Le solveur ainsi que le recuit simulé obtiennent une nette amélioration avec un score de 72.62. Bien évidemment, pour des instances à petites échelles le recuit simulé prend plus de temps que le solveur.

Ainsi, le glouton pose problème puisque bien que nous somme sur des instances jugées faciles il tombe naîvement dans le piège et obtient un haut score.
Le recuit simulé et le solveur apportent une petite amélioration seulement.


### 4.2 Instance Facile — Solveur Exact (Branch and Bound)

Paramètres :

| Paramètre | Valeur |
| --- | --- |
| Température initiale | 1000.0 |
| Température finale | 0.1 |
| Facteur de refroidissement | 0.995 |
| Nombre d'itérations par palier | 200 |

Voici les résultats obtenus pour **3 villes** et **0 arcs bloqués** :

**Algorithme Glouton :**
- Cycle : `[0, 2, 1, 0]`
- Score : **97.23**

**Solveur Exact (Branch and Bound) :**
- Cycle optimal : `[0, 2, 1, 0]`
- Score optimal : **97.23**

**Algorithme Recuit Simulé :**
- Cycle : `[0, 2, 1, 0]`
- Score : **97.23**

Sur cette instance très simple (3 villes), les **trois algorithmes** trouvent exactement la même solution. Cela signifie que :

- Le problème est très simple (peu de villes, donc peu de combinaisons)
- Le glouton trouve déjà la solution optimale
- Le recuit simulé n'améliore pas car il n'y a rien à améliorer
- Le solveur exact confirme que cette solution est bien optimale

### 4.3 Instance Difficile — Recuit Simulé

Voici les résultats obtenus pour **74 villes** et **105 arcs bloqués** :

**Algorithme Glouton :**
- Cycle : `[0, 2, 4, 28, 55, 11, 36, 18, 44, 62, 68, 14, 13, 25, ...]`
- Score : **64 896.90**

**Algorithme Recuit Simulé :**
- Cycle : `[0, 49, 48, 22, 67, 35, 51, 26, 61, 37, 24, 15, ...]`
- Score : **61 738.54**

Cette instance est considérée comme **difficile** car :

- Il y a beaucoup de villes (74), donc énormément de solutions possibles
- Un nombre accru d'arcs bloquées et des fenêtres temporelles dures à respecter obligent à visiter les villes dans un ordre très précis

Le recuit simulé améliore la solution en utilisant la méthode 2-opt pour tester de nouveaux chemins, et trouve progressivement un chemin qui respecte mieux les fenêtres temporelles. Le score passe de **64 896.90** (glouton) à **61 738.54** (recuit).

## V. Plan d'expérience

### 5.1 Expérience n°1 — Variation du Facteur de Refroidissement $\alpha$

**Paramètres fixés :**

| Paramètre | Valeur |
| --- | --- |
| Température initiale | 200.0 |
| Température finale | 1.0 |
| Nombre d'itérations par palier | 50 |
| Taille | $n = 30$ villes |

**Résultats :**

| Alpha | Algo | Moyenne | Écart-type | Min | Max | Temps (ms) | Amélioration |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 0.95 | Glouton | 24 968.99 | 1 372.14 | 21 558.40 | 28 521.94 | 2.7 | |
| | Recuit | 22 965.30 | 995.67 | 21 487.35 | 25 562.34 | 1 055.4 | 8.0% |
| 0.99 | Glouton | 24 968.99 | 1 372.14 | 21 558.40 | 28 521.94 | 2.6 | |
| | Recuit | 22 100.98 | 847.07 | 20 474.38 | 23 590.62 | 5 227.3 | 11.5% |
| 0.995 | Glouton | 24 968.99 | 1 372.14 | 21 558.40 | 28 521.94 | 2.7 | |
| | Recuit | 21 933.11 | 1 026.29 | 19 460.88 | 24 477.16 | 10 422.4 | 12.2% |

**Analyse :** On observe que le recuit simulé améliore les résultats du glouton pour toutes les valeurs de α.

Cependant, l’impact du paramètre α est très significatif sur trois facteurs :

-- la qualité de la solution
-- la stabilité (écart-type)
-- le temps de calcul

On observe que la qualité de la solution s’améliore lorsque α augmente jusqu’à 0.995, mais se dégrade légèrement pour α = 0.999.
Cela montre qu’il existe un résultat optimal pour le paramètre α, ici autour de 0.995.

En prenant $\text{Opt}(D) \approx 21 933.11$ comme référence :

$$\text{Performance glouton} = \frac{24968.99}{21933.11} \approx 1.1384 \quad (\text{+13.84\% au-dessus de l'optimum})$$

$$\text{Performance recuit } (\alpha = 0.95) = \frac{22965.30}{21933.11} \approx 1.0471 \quad (\text{+4.71\%})$$

$$\text{Performance recuit } (\alpha = 0.99) = \frac{22100.98}{21933.11} \approx 1.0077 \quad (\text{+0.77\%})$$

$$\text{Performance recuit } (\alpha = 0.995) = \frac{21933.11}{21933.11} = 1.000 \quad (\text{référence})$$

Le facteur α contrôle la vitesse de refroidissement :

Plus α est faible (0.95) :

- refroidissement rapide
- exploration limitée
- rapide mais moins optimal

Plus α est élevé (0.995) :
- refroidissement lent 
- exploration approfondie
- meilleure solution mais très long (Par exemple alpha = 0.995)

Mais si α est trop élevé (0.999), le temps devient très long, sans amélioration significative

Nous pouvons donc conclure que le recuit simulé reste toujours meilleur que le glouton, quel que soit le alpha. Cependant, le choix de α influence fortement les performances.
Un α trop faible limite l’exploration, tandis qu’un α trop élevé augmente fortement le temps de calcul sans gain significatif.
Le meilleur compromis observé est pour α = 0.995.

### 5.2 Expérience n°2 — Variation du Nombre d'Itérations

**Paramètres fixés :**

| Paramètre | Valeur |
| --- | --- |
| Température initiale | 200.0 |
| Température finale | 1.0 |
| Facteur de refroidissement | 0.995 |
| Taille | $n = 30$ villes |

**Résultats :**

| Itérations | Algo | Moyenne | Écart-type | Temps moyen (ms) | Amélioration |
| --- | --- | --- | --- | --- | --- |
| 20 | Recuit | 22376.54 | 988.31 | 3991.1 | 10.4% |
| 30 | Recuit | 22105.33 | 996.41 | 5532 | 11.5% |
| 50 | Recuit | 21933.11 | 1026.29 | 9323.4 | 12.2% |

Dans cette deuxième expérience, on fait varier le nombre d’itérations de l’algorithme du recuit simulé pour voir son impact sur les résultats.

On remarque que la qualité des solutions s'améliore lorsque nous augmentons le nommbre d'itérations. La moyenne des coûts change également selon le nombre d’itérations, et les différences sont notables.

**Analyse :** La qualité des solutions reste stable. En prenant $\text{Opt}(D) \approx 21933.11$ :

$$\frac{22376.54}{21933.11} \approx 1.0202 \quad (\text{+2.02\% avec 20 itérations})$$

$$\frac{22105.33}{21933.11} \approx 1.0079 \quad (\text{+1.0079\% avec 30 itérations})$$

Nous pouvons donc observer que le meilleur résultat est obtenu avec 50 itérations. Les écarts sont forts. De plus augmenter les itérations garantit une nette amélioration du résultat.

Ainsi, le recuit simulé est sensible au nombre d’itérations dans cette plage.

### 5.3 Expérience n°3 — Variation de la Graine Aléatoire

**Paramètres fixés :**

| Paramètre | Valeur |
| --- | --- |
| Température initiale | 200.0 |
| Température finale | 1.0 |
| Facteur de refroidissement | 0.995 |
| Nombre d'itérations | 50 |
| Taille | $n = 30$ villes |

**Résultats :**

| Graine | Algo | Moyenne | Écart-type | Min | Max | Temps (ms) | Amélioration |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 13 | Recuit | 21 933.11 | 1 026.29 | 19 460.88 | 24 477.16 | 9 367.1 | 12.2% |
| 52 | Recuit | 22 135.57 | 869.17 | 19 507.18 | 24 477.16 | 8 370.2 | 10.1% |
| 78 | Recuit | 21 843.29 | 884.41 | 19 479.69 | 23 552.18 | 8 373.7 | 10.2% |

Dans cette expérience, on observe que les résultats du recuit simulé restent très proches d’une graine à l’autre.

Les moyennes varient légèrement entre 21 843 et 22 135
Les écarts-types restent du même ordre de grandeur
Les temps d’exécution sont également très proches

L’amélioration par rapport au glouton reste comprise entre 10.1% et 12.2%, ce qui montre une bonne stabilité des performances

**Analyse :** En prenant $\text{Opt}(D) \approx 21 843.29$ (meilleure solution) :

$$\frac{21843.29}{21843.29} \approx 1.0000 \quad (\text{référence, graine 78})$$

$$\frac{21933.11}{21843.29} \approx 1.0041 \quad (\text{+0.41\%, graine 13})$$

$$\frac{22135.57}{21843.29} \approx 1.0134 \quad (\text{+1.34\%, graine 52})$$

Donc, cette expérience montre que la graine aléatoire n’a pas d’impact significatif sur la qualité globale des solutions.

Le recuit simulé conserve une amélioration stable par rapport au glouton, ce qui confirme sa robustesse face aux différentes instances générées.

On peut conclure que changer la graine aléatoire n’a pas beaucoup d’impact sur la qualité des solutions obtenues.

### 5.4 Expérience n°4 — Variation de la Température Initiale $T_{\text{initial}}$

**Paramètres fixés :**

| Paramètre | Valeur |
| --- | --- |
| Température finale | 1.0 |
| Facteur de refroidissement | 0.995 |
| Nombre d'itérations | 50 |
| Taille | $n = 30$ villes |

**Résultats :**

| $T_{\text{initial}}$ | Algo | Moyenne | Écart-type | Min | Max | Temps (ms) | Amélioration |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 50 | Recuit | 22 167.86 | 830.70 | 20 483.95 | 24 491.27 | 605.5 | 11.2% |
| 100 | Recuit | 22 101.46 | 885.27 | 19 501.52 | 23 540.53 | 687.3 | 11.5% |
| 200 | Recuit | 21 933.11 | 1 026.29 | 19 460.88 | 24 477.16 | 814.6 | 12.2% |
| 500 | Recuit | 22 007.46 | 1 002.59 | 19 460.54 | 24 595.32 | 956.1 | 11.9% |

Dans cette expérience, nous faisons varier la température initiale T_initial dans le but d’analyser son impact sur l’exploration du recuit simulé.

On observe que le recuit simulé améliore systématiquement les résultats du glouton, quelle que soit la valeur de la température initiale.
Cependant, l’impact de T_initial reste modéré. En effet, la qualité des solutions s’améliore légèrement jusqu’à T = 200 au delà (T = 500), nous pouvons observer qu'il n'y a plus de gain significatif et le temps de calcul augmente progressivement

Ainsi lorsque le T est faible (50) l'exploration devient limitée, les solutions deviennent moins bonnes. En revanche lorsque T est optimale (200) nous possédons un bon équilibre entre 
exploration limitée ente l'exploration et la convergence des données. Enfin lorsque T est élevée l'exploration est longue et il n'ya pas de gain réel.

**Analyse :** En prenant $\text{Opt}(D) \approx 21 933.11$ :

$$\frac{22101.46}{21933.11} \approx 1.0077 \quad (\text{+0.77\% pour } T = 100)$$

$$\frac{21933.11}{21933.11} = 1 \quad (\text{référence pour } T = 200)$$

$$\frac{22007.46}{21933.11} \approx 1.0034 \quad (\text{+0.34\% pour } T = 500)$$

La performance s'améliore jusqu'à $T = 200$, puis diminue légèrement. Le **meilleur compromis** est $T_{\text{initial}} \approx 200$.

Donc, la performance du recuit s’améliore lorsque T_initial augmente jusqu’à 200. En efft le le meilleur résultat est atteint pour cette température initiale. Cela nous confirme qu'une température faible limite l’exploration
et une température trop élevée n’apporte pas de gain significatif

### 5.5 Expérience n°5 — Variation du Nombre de Villes

**Paramètres fixés :**

| Paramètre | Valeur |
| --- | --- |
| Température initiale | 200.0 |
| Température finale | 1.0 |
| Facteur de refroidissement | 0.995 |
| Nombre d'itérations | 50 |
| Nombre de runs | 30 |

**Résultats :**

| $n$ (villes) | Algorithme | Score  | Temps (ms) | 
| ------------ | ---------- | ----------: | ---------: | 
| 5            | Solveur    |     1206.66 |      112.7 |   
|              | Recuit     |     1624.22 (moyenne)|     1591.5 (moyenne) |     
| 8            | Solveur    |     3221.48 |     2244.9 |           
|              | Recuit     |     3493.42 (moyenne) |     2698.5 (moyenne) |        
| 10           | Solveur    |     6339.27 |    56354.3 |          
|              | Recuit     |     5017.51 (moyenne) |     2930.2 (moyenne)|      


On observe plusieurs phénomènes importants. Tout d’abord, lorsque le nombre de villes augmente, le score moyen augmente également. Cela est logique puisque l’augmentation du nombre de villes entraîne une distance totale plus importante à parcourir.

On remarque également que le temps d’exécution du solveur exact augmente très fortement avec la taille du problème. Pour 5 villes, le solveur reste rapide, mais à partir de 10 villes le temps devient très élevé. Cela s’explique par la complexité exponentielle du Branch and Bound, qui doit explorer un très grand nombre de solutions possibles.

À l’inverse, le recuit simulé conserve un temps d’exécution beaucoup plus stable lorsque la taille augmente. Bien qu’il ne garantisse pas l’optimalité, il permet d’obtenir rapidement des solutions de bonne qualité même sur des instances plus grandes.

Enfin, on constate que le solveur exact obtient les meilleurs résultats sur les petites instances, puisqu’il est capable de trouver la solution optimale. Cependant, lorsque la taille augmente, le recuit simulé devient plus intéressant en pratique grâce à son temps d’exécution beaucoup plus faible tout en conservant des scores compétitifs.

## VI. Interprétation des Résultats

### 6.1 Rôle du Facteur $\alpha$

On observe que le paramètre α (facteur de refroidissement) joue un rôle essentiel dans le fonctionnement du recuit simulé.

Lorsque α est élevé (proche de 1), la température diminue lentement. L’algorithme explore alors davantage de solutions et accepte plus facilement des solutions moins bonnes de manière temporaire. Cela lui permet d’éviter de rester bloqué dans un minimum local et d’augmenter ses chances de trouver une meilleure solution globale. En revanche, cette exploration plus approfondie entraîne un temps de calcul beaucoup plus important.

À l’inverse, lorsque α est plus faible, la température diminue rapidement. L’algorithme se stabilise plus vite et explore moins de solutions. Il devient donc plus rapide, mais il risque de rester bloqué dans une solution de qualité moyenne.

Ainsi, α permet de contrôler un compromis fondamental :

un α élevé → meilleure qualité de solution mais temps de calcul élevé
un α faible → exécution rapide mais solution moins optimale


Les différentes expériences montrent que :

Le nombre d’itérations a un impact important sur la qualité des solutions, puisque celles-ci s’améliorent à mesure qu’il augmente, même si le temps de calcul croît également.

La température initiale (T_initial) a un impact modéré sur des valeurs trop faible qui limite l’exploration, tandis qu’une valeur trop élevée n’apporte pas d’amélioration significative mais augmente le temps de calcul.

Le meilleur compromis observé est T_initial ≈ 200.

La graine aléatoire a très peu d’influence sur les performances globales car les résultats restent stables, ce qui montre que le recuit simulé est robuste face à la variabilité des instances.

Lorsque le nombre de villes augmente, le coût des solutions augmente fortement ce qui est logique dans notre cas.
L’amélioration apportée par le recuit diminue progressivement :
12% pour n = 30
4% pour n = 60
3% pour n = 80

Ainsi, le recuit simulé reste meilleur que le glouton, mais son avantage diminue lorsque la taille du problème augmente.

Les performances dépendent également des instances générées aléatoirement puisque certaines instances sont simples (peu de contraintes) et d’autres sont plus complexes (routes bloquées, contraintes temporelles).

Le glouton est donc très sensible à ces variations car il fait des choix locaux sans remise en question.

Le recuit simulé, en revanche, est plus robuste car il explore plusieurs solutions et il peut corriger une mauvaise solution initiale.

Cela se traduit par un écart-type plus faible aissi que des résultats plus stables.

Nous avons réalisé 30 runs pour chaque expérience dans le but d’obtenir des résultats fiables.

En effet, comme les instances sont générées aléatoirement, un seul test ne serait pas représentatif. Il pourrait correspondre à un cas particulier (facile ou difficile).

En répétant les expériences, on obtient :

une moyenne correpondant à la performance globale de notre algorithme
un écart-type correpondant à la stabilité de notre algorithme
un minimum et un maximum correpondant à la variabilité de notre algorithme

Cela permet de réduire l’impact du hasard et d’obtenir une analyse plus rigoureuse.

## VII. Étude des Statistiques

### 7.1 Synthèse par Expérience

**Expérience n°1 ($\alpha$) :**

Ici, la taille est fixée (n = 30), ce qui permet d’isoler l’effet du paramètre α.

On observe que :

lorsque α augmente, le score moyen diminue légèrement car les solutions sont meilleures,
l’écart-type diminue globalement puisque les résultats deviennent plus stables
le temps de calcul augmente fortement

Ainsi α est un paramètre clé du recuit simulé. Le meilleur compromis que nous pouvons obtenir est autour de α = 0.995.

**Expérience n°2 (itérations) :**

Dans cette expérience, nous avons fait varier le nombre d’itérations par palier.

Nous observons que :

la moyenne des résultats diminue lorsque le nombre d'itérations augmente,
le temps de calcul augmente avec le nombre d’itérations.

Le paramètre iterations a un impact modéré sur la qualité de nos solutions dans cette plage. Il permet d’améliorer progressivement les résultats, mais les gains deviennent de plus en plus faibles. Pour trouver un bon compromis, nous choisissons 50 itérations.

**Expérience n°3 (graine) :**

La graine aléatoire permet de générer différentes instances du problème.

On observe que :

les résultats du recuit restent globalement proches d’une graine à l’autre (moyenne comprise entre 21 843 et 22 135),
l’amélioration reste stable (≈ 10.1% à 12.2%),
les variations de moyenne et d’écart-type sont faibles.

Ainsi, la graine a peu d’impact sur la qualité globale des résultats. Le recuit simulé fournit des performances stables et qui sont également reproductibles, ce qui nous est essentiel.

**Expérience n°4 ($T_{\text{initial}}$) :**

On observe que lorsque le nombre de villes augmente, le score moyen des deux algorithmes augmente fortement. Cela est logique : plus il y a de villes à visiter, plus la distance totale de la tournée est élevée.

Cependant, l’élément le plus intéressant est l’évolution de l’écart-type.

On remarque que :

l’écart-type du glouton est plus élevé, surtout lorsque la taille augmente,
le recuit simulé présente un écart-type plus faible.

Cela signifie que le glouton est très sensible à l’instance générée (positions des villes, routes bloquées, contraintes). Certaines configurations peuvent fortement dégrader sa performance.

À l’inverse, le recuit simulé est plus robuste : il est capable de corriger une mauvaise solution initiale en explorant d’autres possibilités.

Ainsi la taille de l’instance influence directement la qualité des solutions et la capacité du recuit à converger vers une bonne solution.

**Expérience n°5 (taille) :**

Dans cette expérience, on fait varier la taille de l’instance (n = 5, 8, 10).

On observe que :

les scores augmentent avec la taille (comportement attendu),
le temps de calcul du solveur exact augmente fortement lorsque le nombre de villes augmente,
le recuit simulé garde un temps de calcul plus stable.

Comparaison des performances :

pour n = 5, le solveur exact obtient un score de 1206.66 contre 1624.22 pour le recuit ;
pour n = 8, le solveur exact obtient un score de 3221.48 contre 3493.42 pour le recuit ;
pour n = 10, le recuit obtient un score de 5017.51 contre 6339.27 pour le solveur exact.

Le solveur exact reste très performant sur les petites instances, mais son temps d’exécution devient très important lorsque la taille augmente.

## VIII. Conclusions et Recommandations

Le choix des paramètres du recuit simulé dépend directement de l'objectif : **qualité de la solution** ou **rapidité d'exécution**.

Pour **privilégier la qualité**, nous recommandons :

| Paramètre | Valeur recommandée |
| --- | --- |
| $\alpha$ | ≈ 0.995 (élevé) |
| $T_{\text{initial}}$ | ≈ 200.0 |
| iterations | ≈ 50 |

Ces paramètres permettent :

- Une exploration approfondie
- D'éviter les minima locaux
- D'obtenir les meilleures performances

**L'inconvénient** est le temps de calcul élevé.

> **Conclusion :** Le recuit simulé est un algorithme flexible dont les performances dépendent fortement de ses paramètres. Il n'existe pas de "meilleur réglage universel" le choix dépend du contexte et des contraintes de notre application.

## IX. Code Source — Implémentation Complète

Le code suivant implémente l'ensemble des modules décrits dans ce livrable.

In [10]:
import sys
!{sys.executable} -m pip install pulp


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import csv
import math
import random
import time
import tkinter as tk
import numpy as np
import matplotlib.pyplot as plt
import pulp
from tkinter import filedialog

# -----------------------------------------------------------------------------
# 1. Les classes de base
# -----------------------------------------------------------------------------

class Ville:
    
    def __init__(self, id, x, y, earliest, latest):
        self.id = id
        self.x = x
        self.y = y
        self.earliest = earliest
        self.latest = latest


class Graphe:
   
    def __init__(self, villes):
        self.villes = villes
        self.n = len(villes)
        self.bloquees = set()

    def distance(self, i, j):
        
        v1, v2 = self.villes[i], self.villes[j]
        return math.hypot(v1.x - v2.x, v1.y - v2.y)

    def accessible(self, i, j):
      
        return (i, j) not in self.bloquees


# -----------------------------------------------------------------------------
# 2. Génération d'instances aléatoires
# -----------------------------------------------------------------------------

def generer_instance(n, ratio_bloquees=0.02, graine=None):
  
    if graine is not None:
        random.seed(graine)
        np.random.seed(graine)

    villes = []

    # Le dépôt, toujours au centre, fenêtre très large pour ne jamais bloquer
    villes.append(Ville(0, 50.0, 50.0, 0, 10000))

    # Les clients : position et fenêtre horaire aléatoires
    for i in range(1, n):
        x = round(random.uniform(0, 100), 2)
        y = round(random.uniform(0, 100), 2)
        earliest = round(random.uniform(0, 50), 1)
        latest   = round(earliest + random.uniform(20, 100), 1)
        villes.append(Ville(i, x, y, earliest, latest))

    g = Graphe(villes)

    # On bloque quelques routes entre clients (2% par défaut)
    paires = [(i, j) for i in range(1, n)
                     for j in range(1, n) if i != j]
    if paires:
        nb_bloquees = int(len(paires) * ratio_bloquees)
        for (i, j) in random.sample(paires, nb_bloquees):
            g.bloquees.add((i, j))

    return g


# -----------------------------------------------------------------------------
# 3. Évaluation d'une tournée
# -----------------------------------------------------------------------------

def evaluer_tournee(tour, g):
    
    temps    = 0
    cout     = 0
    penalite = 0

    for i in range(len(tour) - 1):
        a, b = tour[i], tour[i + 1]

        # Si la route est bloquée, on pénalise mais on passe quand même
        if not g.accessible(a, b):
            penalite += 1000

        d = g.distance(a, b)
        cout  += d
        temps += d

        # Arrivée trop tôt → on attend (gratuit)
        if temps < g.villes[b].earliest:
            temps = g.villes[b].earliest

        # Arrivée trop tard → pénalité
        if temps > g.villes[b].latest:
            penalite += 1000

    return cout + penalite


# -----------------------------------------------------------------------------
# 4. Algorithme glouton — le plus proche voisin
# -----------------------------------------------------------------------------

def glouton(g):

    non_visite = list(range(1, g.n))
    tour = [0]

    while non_visite:
        courant = tour[-1]

        # On préfère les routes libres, mais si tout est bloqué on prend quand même
        accessibles = [x for x in non_visite if g.accessible(courant, x)]
        candidats   = accessibles if accessibles else non_visite

        # La ville la plus proche parmi les candidats
        prochain = min(candidats, key=lambda x: g.distance(courant, x))
        tour.append(prochain)
        non_visite.remove(prochain)

    tour.append(0)  # Retour au dépôt pour fermer le cycle
    return tour


# -----------------------------------------------------------------------------
# 5. Solveur exact — Branch and Bound via PuLP
# -----------------------------------------------------------------------------

def solveur_exact(g, PENALITE=1000):

    n         = g.n
    customers = list(range(1, n))
    START, END = "start", "end"

    # Fenêtres temporelles pour tous les nœuds (dépôt dédoublé)
    e = {i: g.villes[i].earliest for i in customers}
    l = {i: g.villes[i].latest   for i in customers}
    e[START], l[START] = 0, g.villes[0].latest
    e[END],   l[END]   = 0, l[START] + 10000

    def d(i, j):
        # START et END correspondent tous les deux au dépôt physique (ville 0)
        ri = 0 if i in (START, END) else i
        rj = 0 if j in (START, END) else j
        return g.distance(ri, rj)

    prob = pulp.LpProblem("TSPTW", pulp.LpMinimize)

    # Variables binaires pour chaque arc possible
    x = {}
    for j in customers:
        x[START, j] = pulp.LpVariable(f"x_start_{j}", cat="Binary")
    for i in customers:
        x[i, END] = pulp.LpVariable(f"x_{i}_end", cat="Binary")
    for i in customers:
        for j in customers:
            if i != j:
                x[i, j] = pulp.LpVariable(f"x_{i}_{j}", cat="Binary")

    # Temps d'arrivée en chaque ville — pas borné par latest pour garder les TW souples
    s = {i: pulp.LpVariable(f"s_{i}", lowBound=0) for i in customers}
    s[START] = pulp.LpVariable("s_start", lowBound=0)
    s[END]   = pulp.LpVariable("s_end",   lowBound=0)

    # Variables pour détecter les violations de fenêtres temporelles
    retard = {i: pulp.LpVariable(f"ret_{i}", lowBound=0) for i in customers}
    v_tw   = {i: pulp.LpVariable(f"vtw_{i}", cat="Binary") for i in customers}

    # Variables pour détecter les arcs bloquées empruntées
    v_bloc = {}
    for (i, j) in g.bloquees:
        if (i, j) in x:
            v_bloc[i, j] = pulp.LpVariable(f"vb_{i}_{j}", cat="Binary")

    # Ce qu'on veut minimiser : distance + pénalités TW + pénalités arcs bloquées
    prob += (
        pulp.lpSum(d(i, j) * x[i, j] for (i, j) in x)
        + PENALITE * pulp.lpSum(v_tw[i]   for i in customers)
        + PENALITE * pulp.lpSum(v_bloc[k] for k in v_bloc)
    )

    # C1 — Chaque ville doit être visitée exactement une fois
    for j in customers:
        prob += pulp.lpSum(x[i, j] for (i, jj) in x if jj == j) == 1

    # C2 — On part du dépôt une fois et on y revient une fois
    prob += pulp.lpSum(x[START, j] for j in customers) == 1
    prob += pulp.lpSum(x[i, END]   for i in customers) == 1

    # C3 — Si on entre dans une ville, on en repart (conservation du flot)
    for h in customers:
        prob += (
            pulp.lpSum(x[i, j] for (i, j) in x if j == h)
            == pulp.lpSum(x[i, j] for (i, j) in x if i == h)
        )

    # C4 — Si on emprunte une route bloquée, v_bloc s'active (pénalité)
    for (i, j) in v_bloc:
        prob += v_bloc[i, j] >= x[i, j]

    # C5 — On part du dépôt à t=0
    prob += s[START] == 0

    # C5 — Propagation des temps : si on va de i à j, s[j] >= s[i] + distance
    # Le Big-M désactive la contrainte quand on n'emprunte pas cet arc
    for (i, j) in x:
        M = 10000
        prob += s[j] >= s[i] + d(i, j) - M * (1 - x[i, j])

    # C6 — Gestion des fenêtres temporelles en mode souple
    for i in customers:
        prob += s[i] >= e[i]

        M_i = 10000
        prob += retard[i] >= s[i] - l[i]
        prob += retard[i] <= M_i * v_tw[i]
        prob += v_tw[i]   >= retard[i] / M_i  

    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    status = pulp.LpStatus[prob.status]
    if status not in ("Optimal", "Feasible"):
        return None, float("inf")

    succ = {
        i: j for (i, j), var in x.items()
        if pulp.value(var) is not None and pulp.value(var) > 0.5
    }

    tour = [0]
    cur  = START
    for _ in range(n):
        cur = succ.get(cur)
        if cur is None or cur == END:
            break
        tour.append(cur)
    tour.append(0)

    if len(tour) != n + 1:
        return None, float("inf")

    return tour, evaluer_tournee(tour, g)


# -----------------------------------------------------------------------------
# 6. Recuit simulé 
# -----------------------------------------------------------------------------

def recuit(g, t_initial=200.0, t_final=0.1, alpha=0.995, iterations=50):
   
    # On commence avec la solution du glouton
    tour           = glouton(g)
    meilleur       = tour[:]
    meilleur_score = evaluer_tournee(tour, g)
    score_actuel   = meilleur_score
    T              = t_initial
    historique     = []

    while T > t_final:
        for _ in range(iterations):

            # Mouvement 2-opt : on choisit deux points i et j et on inverse
            # le segment du tour entre eux
            nouveau = tour[:]
            i = random.randint(1, g.n - 2)
            j = random.randint(i + 1, g.n - 1)
            nouveau[i:j+1] = nouveau[i:j+1][::-1]

            score = evaluer_tournee(nouveau, g)
            delta = score - score_actuel

            # Critère de Metropolis : accepter si c'est mieux,
            # ou accepter quand même avec une certaine probabilité si c'est pire
            if delta < 0 or random.random() < math.exp(-delta / T):
                tour         = nouveau
                score_actuel = score
                if score_actuel < meilleur_score:
                    meilleur       = tour[:]
                    meilleur_score = score_actuel

        historique.append(meilleur_score)
        T *= alpha  

    return meilleur, historique


# -----------------------------------------------------------------------------
# 7. Vérification de la solution
# -----------------------------------------------------------------------------

def verifier_solution(tour, g, nom_algo="Algorithme"):
  
    villes_visitees = tour[1:-1]
    uniques         = set(villes_visitees)

    print(f"  [{nom_algo}] Nombre de visites : {len(villes_visitees)}")
    print(f"  [{nom_algo}] Nombre unique     : {len(uniques)}")
    print(f"  [{nom_algo}] Nombre attendu    : {g.n - 1}")

    if len(uniques) != g.n - 1 or len(villes_visitees) != g.n - 1:
        print(f"  [{nom_algo}] ERREUR : villes manquantes ou doublons détectés")
    else:
        print(f"  [{nom_algo}] Cycle hamiltonien valide")

    # On compte les arcbloquées empruntées
    violations_bloc = [
        (tour[k], tour[k+1])
        for k in range(len(tour) - 1)
        if (tour[k], tour[k+1]) in g.bloquees
    ]
    nb_bloc = len(violations_bloc)
    if nb_bloc > 0:
        print(f"  [{nom_algo}] arcs bloquées empruntées : {nb_bloc} "
              f"→ pénalité = {nb_bloc * 1000} | arcs : {violations_bloc}")
    else:
        print(f"  [{nom_algo}] Arêtes bloquées empruntées : 0 → aucune pénalité")

    # On recompte les violations de fenêtres temporelles en resimulant le trajet
    temps = 0
    nb_tw = 0
    for k in range(len(tour) - 1):
        d = g.distance(tour[k], tour[k+1])
        temps += d
        b = tour[k+1]
        if temps < g.villes[b].earliest:
            temps = g.villes[b].earliest  
        if temps > g.villes[b].latest:
            nb_tw += 1  

    if nb_tw > 0:
        print(f"  [{nom_algo}] Violations fenêtres temporelles : {nb_tw} "
              f"→ pénalité = {nb_tw * 1000}")
    else:
        print(f"  [{nom_algo}] Violations fenêtres temporelles : 0 → aucune pénalité")

    print(f"  [{nom_algo}] Pénalité totale : {(nb_bloc + nb_tw) * 1000}")






In [ ]:
# -----------------------------------------------------------------------------
# 8. Programme principal
# -----------------------------------------------------------------------------

if __name__ == "__main__":

    print("=" * 55)
    print("  TESTS UNITAIRES DES CONTRAINTES")
    print("=" * 55)

    # Test 1 : on crée une ville avec une fenêtre impossible (latest = 0.001)
    print("\n----- TEST CONTRAINTE 1 : Fenêtres Temporelles -----")
    villes_tw = [
        Ville(0, 0, 0, 0, 1000),
        Ville(1, 10, 0, 0, 1000),
        Ville(2, 50, 0, 0, 0.001),  
    ]
    g_tw = Graphe(villes_tw)
    score_sans = evaluer_tournee([0, 1, 0],    g_tw)  # on ne passe pas par la ville 2
    score_avec = evaluer_tournee([0, 1, 2, 0], g_tw)  
    print(f"  Score sans violation TW  : {score_sans:.2f}")
    print(f"  Score avec violation TW  : {score_avec:.2f}")
    if score_avec > score_sans + 999:
        print("  Contrainte TW active : pénalité de 1000 bien appliquée")
    else:
        print("  Contrainte TW non détectée")

    # Test 2 : on bloque la route (1→2) et on vérifie que ça coûte 1000 de plus
    print("\n----- TEST CONTRAINTE 2 : Arcs Bloquées -----")
    villes_bl = [
        Ville(0, 0, 0, 0, 1000),
        Ville(1, 10, 0, 0, 1000),
        Ville(2, 20, 0, 0, 1000),
    ]
    g_bl     = Graphe(villes_bl)
    tour_bl  = [0, 1, 2, 0]
    score_libre  = evaluer_tournee(tour_bl, g_bl)      
    g_bl.bloquees.add((1, 2))
    score_bloque = evaluer_tournee(tour_bl, g_bl)      
    print(f"  Score sans arc bloquée : {score_libre:.2f}")
    print(f"  Score avec arc bloquée : {score_bloque:.2f}")
    if score_bloque > score_libre + 999:
        print("  Contrainte arc bloquée active : pénalité de 1000 bien appliquée")
    else:
        print("  Contrainte arc bloquée non détectée")

    # ══════════════════════════════════════════════════════════════════════════
    # Étape 2 — Résolution
    # ══════════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 55)
    print("  RÉSOLUTION — CYCLE HAMILTONIEN")
    print("=" * 55)

    # Pour imposer une taille : mettre un entier. Pour aléatoire : None
    n_impose = None

    if n_impose is not None:
        n_instance = n_impose
        print(f"\nNombre de villes imposé : {n_instance}")
    else:
        n_instance = random.randint(3, 100)
        print(f"\nNombre de villes tiré aléatoirement : {n_instance}")

    g = generer_instance(n_instance)
    print(f"Instance générée : {g.n} villes, {len(g.bloquees)} arcs bloquées")

    print("\n----- GLOUTON -----")
    t1 = glouton(g)
    print(f"  Cycle : {t1}")
    print(f"  Score : {evaluer_tournee(t1, g):.2f}")

    print("\n----- SOLVEUR EXACT — Branch and Bound -----")
    if g.n <= 10:
        t0 = time.perf_counter()
        tour_exact, score_exact = solveur_exact(g)
        duree = time.perf_counter() - t0
        print(f"  Cycle optimal : {tour_exact}")
        print(f"  Score optimal : {score_exact:.2f}")
        print(f"  Temps         : {duree*1000:.1f} ms")
        score_g = evaluer_tournee(t1, g)
        gap_g   = (score_g - score_exact) / score_exact * 100 if score_exact > 0 else 0
        print(f"  Performance glouton : {score_g/score_exact:.3f} "
              f"(écart : {gap_g:.1f}%)")
    else:
        print(f"  Instance trop grande ({g.n} villes > 10).")
        print(f"  Solveur exact non applicable — utiliser glouton ou recuit.")

    print("\n----- RECUIT SIMULE -----")
    t2, historique = recuit(g)
    print(f"  Cycle : {t2}")
    print(f"  Score : {evaluer_tournee(t2, g):.2f}")

    # Performance relative H(D)/Opt(D) — proche de 1 = proche de l'optimal
    if g.n <= 10 and 'score_exact' in dir():
        score_r = evaluer_tournee(t2, g)
        gap_r   = (score_r - score_exact) / score_exact * 100 if score_exact > 0 else 0
        print(f"  Performance recuit  : {score_r/score_exact:.3f} "
              f"(écart : {gap_r:.1f}%)")

    # Vérification pour les trois algos
    print("\n----- VERIFICATION -----")
    verifier_solution(t1, g, nom_algo="Glouton")
    print()
    verifier_solution(t2, g, nom_algo="Recuit Simulé")
    if g.n <= 10 and 'tour_exact' in dir() and tour_exact is not None:
        print()
        verifier_solution(tour_exact, g, nom_algo="Solveur Exact")

    # ══════════════════════════════════════════════════════════════════════════
    # Étape 3 — Plan d'expérience
    # ══════════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 55)
    print("   PLAN D'EXPÉRIENCE — Tailles variables")
    print("=" * 55)

    tailles = [8]   
    nb_runs = 30   
    for n in tailles:
        scores_g, scores_r, temps_g, temps_r = [], [], [], []

        for seed in range(nb_runs):
            instance = generer_instance(n, graine=seed * 13 + n)

            
            t0 = time.perf_counter()
            t1 = glouton(instance)
            temps_g.append(time.perf_counter() - t0)
            scores_g.append(evaluer_tournee(t1, instance))

         
            t0 = time.perf_counter()
            t2, _ = recuit(instance, t_initial=200.0, t_final=1.0,
                           alpha=0.995, iterations=50)
            temps_r.append(time.perf_counter() - t0)
            scores_r.append(evaluer_tournee(t2, instance))

        # On affiche les stats : moyenne, écart-type, min, max, temps moyen
        print(f"\nn={n} villes ({nb_runs} runs) :")
        print(f"  Glouton — moy: {np.mean(scores_g):.2f}  "
              f"écart-type: {np.std(scores_g):.2f}  "
              f"min: {np.min(scores_g):.2f}  max: {np.max(scores_g):.2f}  "
              f"temps moy: {np.mean(temps_g)*1000:.1f} ms")
        print(f"  Recuit  — moy: {np.mean(scores_r):.2f}  "
              f"écart-type: {np.std(scores_r):.2f}  "
              f"min: {np.min(scores_r):.2f}  max: {np.max(scores_r):.2f}  "
              f"temps moy: {np.mean(temps_r)*1000:.1f} ms")

        # Le pourcentage d'amélioration du recuit par rapport au glouton
        amelio = (np.mean(scores_g) - np.mean(scores_r)) / np.mean(scores_g) * 100
        print(f"  Amélioration recuit vs glouton : {amelio:.1f}%")

    # ══════════════════════════════════════════════════════════════════════════
    # Étape 4 — Graphique final
    # ══════════════════════════════════════════════════════════════════════════
    moy_g, moy_r, std_g, std_r = [], [], [], []

    for n in tailles:
        scores_g, scores_r = [], []
        for seed in range(nb_runs):
            instance = generer_instance(n, graine=seed * 13 + n)
            t1 = glouton(instance)
            scores_g.append(evaluer_tournee(t1, instance))
            t2, _ = recuit(instance, t_initial=200.0, t_final=1.0,
                           alpha=0.995, iterations=50)
            scores_r.append(evaluer_tournee(t2, instance))
        moy_g.append(np.mean(scores_g))
        moy_r.append(np.mean(scores_r))
        std_g.append(np.std(scores_g))
        std_r.append(np.std(scores_r))

    plt.figure(figsize=(10, 5))
    plt.errorbar(tailles, moy_g, yerr=std_g, marker="o", color="#2980b9",
                 label="Glouton", capsize=5, lw=2)
    plt.errorbar(tailles, moy_r, yerr=std_r, marker="s", color="#e74c3c",
                 label="Recuit Simulé", capsize=5, lw=2)
    plt.xlabel("Nombre de villes")
    plt.ylabel("Score moyen")
    plt.title("Comparaison Glouton vs Recuit Simulé\n"
              "(moyenne ± écart-type sur 30 runs)")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig("plan_experience.png", dpi=150, bbox_inches="tight")
    print("\nGraphique sauvegardé : plan_experience.png")
    plt.show()


  TESTS UNITAIRES DES CONTRAINTES

----- TEST CONTRAINTE 1 : Fenêtres Temporelles -----
  Score sans violation TW  : 20.00
  Score avec violation TW  : 1100.00
  Contrainte TW active : pénalité de 1000 bien appliquée

----- TEST CONTRAINTE 2 : Arcs Bloquées -----
  Score sans arc bloquée : 40.00
  Score avec arc bloquée : 1040.00
  Contrainte arc bloquée active : pénalité de 1000 bien appliquée

  RÉSOLUTION — CYCLE HAMILTONIEN

Nombre de villes imposé : 4
Instance générée : 4 villes, 0 arcs bloquées

----- GLOUTON -----
  Cycle : [0, 1, 3, 2, 0]
  Score : 1122.77

----- SOLVEUR EXACT — Branch and Bound -----
  Cycle optimal : [0, 1, 2, 3, 0]
  Score optimal : 1121.50
  Temps         : 435.5 ms
  Performance glouton : 1.001 (écart : 0.1%)

----- RECUIT SIMULE -----
  Cycle : [0, 1, 2, 3, 0]
  Score : 1121.50
  Performance recuit  : 1.000 (écart : 0.0%)

----- VERIFICATION -----
  [Glouton] Nombre de visites : 3
  [Glouton] Nombre unique     : 3
  [Glouton] Nombre attendu    : 3
  [Glou

KeyboardInterrupt: 